# Lecture 9: Eat Your Own Dogfood - Verified Crypto in the Agent's Own Loop

Every lecture so far had the agent consume EVIDENCE about a verified Ed25519 implementation while checking that evidence's signatures with OpenSSL - an unverified implementation of the very primitive the evidence is about. That is a defensible bootstrap, but it leaves an ironic gap. This lecture closes it: pacta can build a verifier binary from the PINNED, PROVEN source workspace - the exact commit the dalek certificates pin, serial backend pinned exactly as the verified extraction pins it - and route its own signature checks through it.


## Learning Objectives

- State precisely which parts of the dogfood verifier are certificate-covered and which are its trusted base.
- Extract a raw Ed25519 key from an OpenSSL PEM by hand (napkin) and mechanically (real).
- Demonstrate backend dispatch and the fail-closed `--require-verified-verifier` policy.
- Defend the hybrid post-quantum posture: one proven-classical signature plus one required-but-honest ML-DSA slot.


## What "verified" means here - the honest ledger

The binary calls `ed25519_dalek::VerifyingKey::verify` in the pinned workspace. The certificates cover `verify_sha512`, the extraction-refactored image of that same path (the delta is the documented hash-wrapper refactor in the pinned source). Certificate-covered: field arithmetic, the group law, scalars, encoding/decoding, constructive decompression, and the four-tier acceptance criterion. Trusted base: SHA-512 (an oracle in the theorems - the proofs hold for whatever bytes it produces), roughly fifteen lines of wire glue, rustc, and the extraction pipeline. The provenance sidecar written at build time records the source commit, the backend cfg, and this exact coverage note - the dogfood claim is itself a claim card.


## Napkin: read a PEM with your eyes

An OpenSSL Ed25519 public key PEM is a base64-wrapped DER SubjectPublicKeyInfo (RFC 8410), and for this one algorithm the DER is FIXED: twelve prefix bytes `302a300506032b6570032100`, then the raw 32-byte key. Decode one by hand:


In [ ]:
from pathlib import Path
import base64, subprocess, sys, tempfile

repo_root = Path.cwd()
if not (repo_root / "src" / "pacta").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from pacta.signing import generate_ed25519_keypair

tmp = Path(tempfile.mkdtemp(prefix="dogfood-lecture-"))
generate_ed25519_keypair(tmp / "k.key", tmp / "k.pub")
pem = (tmp / "k.pub").read_text()
print(pem)
body = "".join(line for line in pem.splitlines() if "-----" not in line)
der = base64.b64decode(body)
print("DER length:", len(der), "(should be 12 + 32 = 44)")
print("prefix:    ", der[:12].hex(), "(the fixed Ed25519 SPKI header)")
print("raw key:   ", der[12:].hex())


In [ ]:
# REAL: the same extraction, mechanically, with validation - and the
# dispatch that prefers the proven-path binary when it exists.
from pacta.dogfood import locate_verifier, pem_public_key_to_raw
from pacta.signing import sign_payload_ed25519, verify_payload_ed25519_detailed

raw = pem_public_key_to_raw(tmp / "k.pub")
assert raw == der[12:]
print("mechanical extraction matches the napkin:", raw.hex()[:16], "...")

payload = b"the agent checks its own evidence"
signature = sign_payload_ed25519(payload, tmp / "k.key")
ok, error, backend = verify_payload_ed25519_detailed(payload, signature, tmp / "k.pub")
print("valid:", ok, "| backend:", backend)
binary = locate_verifier()
print("dogfood binary:", binary or "not built (OpenSSL fallback in effect - a recorded downgrade)")


Build the proven-path verifier once per machine (it needs a local checkout of the pinned source workspace and cargo):

```bash
pacta dogfood-build --source ~/GitClone/FormalVerification/sources/curve25519-dalek-source
pacta dogfood-status
```

With the binary in place, every receipt and attestation check reports `ed25519_backend: verified-dalek-serial`, and policies can DEMAND it:

```bash
pacta receipt-verify ... --require-verified-verifier   # fails closed on OpenSSL fallback
pacta agent ... --require-verified-verifier ...
```


## The post-quantum line, held honestly

The dogfood loop deliberately does NOT extend to ML-DSA. There is no formally verified ML-DSA implementation in this corpus, and pretending otherwise would poison the whole posture. The hybrid strategy is therefore asymmetric on purpose:

- **Ed25519 (classical): proven path.** The signature everyone can check today runs on certificate-covered code.
- **ML-DSA-65 (post-quantum): required, honest, unavailable-until-real.** The tree-head slot exists in every signed structure; `--require-signatures both` fails CLOSED on hosts without a real FIPS 204 backend; and when a real backend lands, the policy flips on without a schema change.

A migration strategy that records "we cannot do this yet" as a deployment blocker is strictly stronger than one that ships a placeholder. Blockers get fixed; placeholders get trusted.


In [ ]:
from pacta.postquantum import detect_ml_dsa

capability = detect_ml_dsa()
print("ml-dsa available:", capability.available)
print("reason:", capability.reason)
print("slot as recorded in every STH:", capability.to_signature_slot())


## Exercises

- Flip one byte of a signature and verify through both backends; confirm both reject and that the BACKEND that rejected is recorded.
- The dogfood binary's trusted base includes rustc. The certificates' trusted base includes Charon/Aeneas. Draw the two trust diagrams side by side; which assumptions are shared?
- Napkin, then real: decode a second PEM by hand; then corrupt its DER prefix and confirm `pem_public_key_to_raw` rejects it.
- Policy design: when should an agent REFUSE to fall back to OpenSSL? Write the deployment rule and its recovery path.
- Research checkpoint: what would a proof-carrying SHA-512 change about the coverage note in the provenance sidecar?
